# 03 · Statistics: multiple regression and hypothesis tests

All on **real** DIKSHA data (`marts.mart_content_features`), student content with usage data.

**Caveats that apply throughout**
- **Observational data:** results are *associations*, not causes.
- **Selection:** only resources with usage data (~82% of student content) — see `02_diagnostics` §3.
- **Portal plays only** (DIKSHA web), not app/offline use.
- **Huge n (~300K):** almost any difference is "statistically significant", so **effect sizes and
  confidence intervals matter more than p-values**.

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT = "learning-reach-473210"
client = bigquery.Client(project=PROJECT, location="asia-south1")

def q(sql: str) -> pd.DataFrame:
    """Run SQL in BigQuery and return a DataFrame (INT64 -> float, see 01_eda)."""
    df = client.query(sql).to_dataframe()
    int_cols = df.select_dtypes(include=["Int64"]).columns
    return df.astype({c: "float64" for c in int_cols})

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({"figure.figsize": (10, 5), "axes.spines.top": False, "axes.spines.right": False})
FOCUS = ["Hindi", "Marathi", "Gujarati", "Kannada", "Punjabi", "Assamese"]

import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests
from scipy import stats
rng = np.random.default_rng(42)

In [ ]:
df = q("""
SELECT identifier, language_group, language, format, grade_band, subject, board_type,
       content_age_days, play_sessions, log_play_sessions, has_usage,
       avg_rating, ratings_count
FROM `learning-reach-473210.marts.mart_content_features`
WHERE NOT is_teacher_content
""")
print(len(df), "student resources;", int(df["has_usage"].sum()), "with usage")

# Keep the main subjects; group the long tail so the model stays readable.
main_subjects = ["Mathematics", "Science", "Social Science", "English", "Hindi",
                 "Environmental Studies", "Other languages"]
df["subject_g"] = np.where(df["subject"].isin(main_subjects), df["subject"], "Other/multiple")
df["log_age"] = np.log1p(df["content_age_days"])

model_df = df[df["has_usage"]
              & (df["language_group"] != "Unknown")
              & (df["format"].isin(["Video", "PDF", "Interactive"]))
              & (df["grade_band"] != "No class tag")
              & (df["board_type"] != "Other / none")].dropna(subset=["log_age"])
print(len(model_df), "rows in the regression")

## 1. Multiple regression (OLS)

```
log(play_sessions + 1) ~ language group + format + grade band + subject + board type + log(content age)
```
Reference categories: English, PDF, Primary (1-5), Mathematics, National board.
Ratings are **not** used as a predictor: they are themselves a result of usage.

On a log outcome, coefficient *b* means roughly **(e^b − 1) × 100 % more plays** than the reference,
holding the other variables fixed. Standard errors are **HC3 heteroskedasticity-robust**.

In [ ]:
formula = ("log_play_sessions ~ C(language_group, Treatment('English')) + C(format, Treatment('PDF'))"
           " + C(grade_band, Treatment('Primary (1-5)')) + C(subject_g, Treatment('Mathematics'))"
           " + C(board_type, Treatment('National')) + log_age")
ols = smf.ols(formula, data=model_df).fit(cov_type="HC3")
print(f"n = {int(ols.nobs):,}   R² = {ols.rsquared:.3f}   adj. R² = {ols.rsquared_adj:.3f}")

table = pd.DataFrame({"coef": ols.params, "ci_low": ols.conf_int()[0], "ci_high": ols.conf_int()[1],
                      "p_value": ols.pvalues})
table["pct_effect"] = (np.exp(table["coef"]) - 1) * 100
table["pct_low"] = (np.exp(table["ci_low"]) - 1) * 100
table["pct_high"] = (np.exp(table["ci_high"]) - 1) * 100
table.index = (table.index.str.replace(r"C\((\w+).*?\)\[T\.", r"\1 = ", regex=True)
                          .str.replace("]", "", regex=False))
display(table[["pct_effect", "pct_low", "pct_high", "p_value"]].drop(index="Intercept").round(1))

In [ ]:
# Effect plot: % difference in plays vs the reference category, with 95% CI
t = table.drop(index=["Intercept", "log_age"]).sort_values("pct_effect")
plt.figure(figsize=(9, 0.35 * len(t) + 1))
plt.errorbar(t["pct_effect"], range(len(t)),
             xerr=[t["pct_effect"] - t["pct_low"], t["pct_high"] - t["pct_effect"]],
             fmt="o", color="#378ADD", ecolor="#888780")
plt.yticks(range(len(t)), t.index)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("% more (or fewer) plays than the reference, other factors equal (95% CI)")
plt.title("What is associated with more usage?")
plt.show()

### Regression diagnostics
- **Residuals vs fitted** (sample): look for strong curvature or funnels.
- **Breusch–Pagan**: tests constant variance — expected to fail at this n; that is why HC3 SEs are used.
- **VIF**: multicollinearity; values above ~5–10 would make individual coefficients unreliable.

In [ ]:
sample = model_df.sample(n=min(20000, len(model_df)), random_state=42)
fitted = ols.predict(sample)
resid = sample["log_play_sessions"] - fitted

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(fitted, resid, s=3, alpha=0.3, color="#378ADD")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("fitted"); axes[0].set_ylabel("residual"); axes[0].set_title("Residuals vs fitted (20K sample)")
sm.qqplot(resid, line="s", ax=axes[1], markersize=2)
axes[1].set_title("Q-Q plot of residuals")
plt.show()

ols_s = smf.ols(formula, data=sample).fit()
bp_stat, bp_p, _, _ = het_breuschpagan(ols_s.resid, ols_s.model.exog)
print(f"Breusch-Pagan (sample): stat = {bp_stat:,.1f}, p = {bp_p:.2g}  -> heteroskedastic: use HC3 SEs")

X = pd.DataFrame(ols_s.model.exog, columns=ols_s.model.exog_names)
vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(X.shape[1])], index=X.columns)
display(vif.drop("Intercept").sort_values(ascending=False).round(2).to_frame("VIF").head(10))

### Sensitivity: who has usage data at all? (logistic regression)
If formats or languages differ strongly in *whether* usage is recorded, the OLS above (usage-only rows)
could be biased. Odds ratios > 1 = more likely to have usage data.

In [ ]:
logit_df = df[(df["language_group"] != "Unknown") & df["format"].isin(["Video", "PDF", "Interactive"])
              & (df["board_type"] != "Other / none")].dropna(subset=["log_age"])
logit_df = logit_df.sample(n=min(100000, len(logit_df)), random_state=42)
logit_df["has_usage_i"] = logit_df["has_usage"].astype(int)
logit = smf.logit("has_usage_i ~ C(language_group, Treatment('English')) + C(format, Treatment('PDF'))"
                  " + C(board_type, Treatment('National')) + log_age", data=logit_df).fit(disp=False)
odds = pd.DataFrame({"odds_ratio": np.exp(logit.params), "low": np.exp(logit.conf_int()[0]),
                     "high": np.exp(logit.conf_int()[1])}).drop(index="Intercept")
odds.round(2)

## 2. Hypothesis tests

Each: hypotheses, test choice (and why), effect size with CI. α = 0.05, **Holm-corrected** across the
three primary tests. Play counts are extremely skewed, so tests 1–2 use **rank-based** tests.

In [ ]:
def rank_biserial(x, y):
    """Effect size for Mann-Whitney: P(x > y) - P(y > x), from -1 to 1."""
    u = stats.mannwhitneyu(x, y, alternative="two-sided").statistic
    return 2 * u / (len(x) * len(y)) - 1

def bootstrap_ci(func, x, y, n_boot=300, size=5000):
    """95% bootstrap CI of an effect size, on subsamples (keeps it fast at this n)."""
    vals = [func(rng.choice(x, min(size, len(x))), rng.choice(y, min(size, len(y))))
            for _ in range(n_boot)]
    return np.percentile(vals, [2.5, 97.5])

results = []

### Test 1 — Regional-language vs English medium (state boards)
- **H0:** plays per resource have the same distribution for regional-language and English-medium content.
- **H1:** they differ.
- **Test:** Mann–Whitney U (two-sided): distributions are heavily skewed, so a t-test on raw plays is inappropriate.
- **Effect size:** rank-biserial correlation r (P(regional > English) − P(English > regional)).

In [ ]:
state = model_df[model_df["board_type"] == "State"]
regional = state.loc[state["language_group"].isin(["Hindi", "Other focus language", "Other regional language"]),
                     "play_sessions"].to_numpy()
english = state.loc[state["language_group"] == "English", "play_sessions"].to_numpy()

u_res = stats.mannwhitneyu(regional, english, alternative="two-sided")
r = rank_biserial(regional, english)
r_ci = bootstrap_ci(rank_biserial, regional, english)
print(f"n regional = {len(regional):,}, n English = {len(english):,}")
print(f"median plays: regional = {np.median(regional):,.0f}, English = {np.median(english):,.0f}")
print(f"U = {u_res.statistic:,.0f}, p = {u_res.pvalue:.2g}, rank-biserial r = {r:.3f} (95% CI {r_ci[0]:.3f} to {r_ci[1]:.3f})")
results.append(("1. Regional vs English (state boards)", u_res.pvalue, f"r = {r:.3f}"))

### Test 2 — Does format matter? (Video vs PDF vs Interactive)
- **H0:** plays per resource have the same distribution across the three formats.
- **H1:** at least one format differs.
- **Test:** Kruskal–Wallis (rank-based ANOVA), then pairwise Mann–Whitney with **Holm** correction as post-hoc.
- **Effect size:** ε² for Kruskal–Wallis; rank-biserial r for each pair.

In [ ]:
groups = {f: model_df.loc[model_df["format"] == f, "play_sessions"].to_numpy()
          for f in ["Video", "PDF", "Interactive"]}
h, p_kw = stats.kruskal(*groups.values())
n_total = sum(len(g) for g in groups.values())
eps2 = (h - len(groups) + 1) / (n_total - len(groups))
print({f: f"median {np.median(g):,.0f} (n={len(g):,})" for f, g in groups.items()})
print(f"Kruskal-Wallis H = {h:,.1f}, p = {p_kw:.2g}, epsilon² = {eps2:.3f}")
results.append(("2. Format (Kruskal-Wallis)", p_kw, f"epsilon² = {eps2:.3f}"))

pairs = [("Interactive", "Video"), ("Interactive", "PDF"), ("Video", "PDF")]
raw_p = [stats.mannwhitneyu(groups[a], groups[b], alternative="two-sided").pvalue for a, b in pairs]
holm_p = multipletests(raw_p, method="holm")[1]
display(pd.DataFrame({"pair": [f"{a} vs {b}" for a, b in pairs],
                      "rank_biserial_r": [rank_biserial(groups[a], groups[b]) for a, b in pairs],
                      "p_holm": holm_p}))

### Test 3 — Ratings: focus languages vs other regional languages
- **H0:** mean rating is equal for focus-language and other-regional-language content.
- **H1:** the means differ.
- **Test:** Welch's t-test (unequal variances). Ratings are bounded 1–5 and averaged over ≥ 10 ratings,
  so means are reasonable here; Welch avoids assuming equal variances.
- **Effect size:** Cohen's d with 95% CI for the mean difference.

In [ ]:
rated = df[(df["ratings_count"] >= 10) & df["avg_rating"].notna()]
focus = rated.loc[rated["language_group"].isin(["Hindi", "Other focus language"]), "avg_rating"].to_numpy()
other = rated.loc[rated["language_group"] == "Other regional language", "avg_rating"].to_numpy()

t_res = stats.ttest_ind(focus, other, equal_var=False)
diff = focus.mean() - other.mean()
se = np.sqrt(focus.var(ddof=1) / len(focus) + other.var(ddof=1) / len(other))
dof = se**4 / ((focus.var(ddof=1) / len(focus))**2 / (len(focus) - 1)
               + (other.var(ddof=1) / len(other))**2 / (len(other) - 1))
ci = diff + np.array([-1, 1]) * stats.t.ppf(0.975, dof) * se
pooled_sd = np.sqrt(((len(focus) - 1) * focus.var(ddof=1) + (len(other) - 1) * other.var(ddof=1))
                    / (len(focus) + len(other) - 2))
d_cohen = diff / pooled_sd
print(f"n focus = {len(focus):,}, n other = {len(other):,}")
print(f"mean rating: focus = {focus.mean():.3f}, other = {other.mean():.3f}, diff = {diff:.3f} (95% CI {ci[0]:.3f} to {ci[1]:.3f})")
print(f"Welch t = {t_res.statistic:.2f}, p = {t_res.pvalue:.2g}, Cohen's d = {d_cohen:.3f}")
results.append(("3. Ratings focus vs other regional", t_res.pvalue, f"d = {d_cohen:.3f}"))

### Multiple comparisons
Three primary tests → Holm correction keeps the family-wise error rate at 5%.
Rule of thumb for effect sizes: |r| or d ≈ 0.1 small, 0.3 medium, 0.5 large; ε² ≈ 0.01 small, 0.06 medium, 0.14 large.

In [ ]:
summary = pd.DataFrame(results, columns=["test", "p_value", "effect_size"])
summary["p_holm"] = multipletests(summary["p_value"], method="holm")[1]
summary["reject_H0"] = summary["p_holm"] < 0.05
summary

## Interpretation (fill in after running)

- **Regression:** holding other factors equal, interactive content gets about …% more plays than PDFs;
  focus-language content …% vs English; … R² = … (most variation is unexplained — expected for usage data).
- **Test 1:** regional vs English: r = … (small/medium) → …
- **Test 2:** format: ε² = … → …
- **Test 3:** ratings: d = … → statistically significant but practically small/large?
- **Caveats:** associations only; usage-data selection; portal plays only.